# 1 · Identify the business needs

To start this project, we first identify the business needs by asking the following question:
What decision does this model support, and who acts on the answer?

# 2· Import all the necessary libraries

The first step of every Jupyter notebook is importing the tools:

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from math import ceil

# 3 · Explore the data

We Upload the train.csv file from the folder directly to this Jupyter notebook to inspect its data properties. We will verify its shape, list its variables and extract some statistical descriptive informations such as central tendency and dispersion.

Before modeling anything, we must take a look at the data presented to us. Exploration tells us what we have, what is broken, what is missing, and which features might carry signal.

In [2]:
train_df=pd.read_csv('train.csv')

print("shape of train.csv DataFrame:",train_df.shape)

shape of this DataFrame: (6534, 39)


In [3]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6534 entries, 0 to 6533
Data columns (total 39 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Total a Pagar                   6469 non-null   float64
 1   Total Bruto                     6469 non-null   float64
 2   Total Liquido                   6469 non-null   float64
 3   Total Imp_                      6469 non-null   float64
 4   Total Portes                    6469 non-null   float64
 5   Total Desc_ Global              6469 non-null   float64
 6   Total Desc_ Linha               6469 non-null   float64
 7   Custo                           6469 non-null   float64
 8   N_ Detalhes                     6469 non-null   float64
 9   invoice_count_current_purchase  6469 non-null   float64
 10  repeat_purchase_90d             6534 non-null   int64  
 11  days_since_previous_purchase    5673 non-null   float64
 12  customer_tenure_days            6469 non-null

A CORRIGIR

Description:

Out of all the 39 columns, each representing a variables (or feature), we count 29 variables of the type float, 3 variables of the type integer and 7 variables of the type string. 

Financial Variables:
0- Total a Pagar: Total amount to pay for this specific transaction.
1- Total Bruto: Gross purchase amount of this specific transaction.
2- Total Liquido: Net purchase amount of this specific transaction.
3- Total Imp_: Total of the tax amount associated with this specific transaction.
4- Total Portes: Total of the shipping charges of this specific transaction.
5- Total Desc_ Global: Total of the global discount applied to the transactions.
6- Total Desc_ Linha: Total of the discounts applied to this specific transaction.
7- Custo: Total cost associated with this specific transaction.
8- N_ Detalhes: Number of detail lines or items recorded on the invoice of this specific transaction.

Transaction's History Variables:
9- invoicecountcurrent_purchase: Number of invoices associated with this current specific transaction.

repeatpurchase90d: Indicator of whether the customer makes a repeat purchase within 90 days. Typically, 1 indicates a repeat purchase and 0 indicates no repeat purchase within that period.
dayssinceprevious_purchase: Number of days between the current purchase and the customer's previous purchase.
customertenuredays: Number of days the customer has been associated with the business, based on the available historical data.
priorpurchasecount: Number of purchases made by the customer before the current transaction.
priortotalspend: Total amount spent by the customer on previous purchases.
priormeanpurchase_value: Average value of the customer's previous purchases.
priorstdpurchase_value: Standard deviation of the customer's previous purchase values, measuring spending variability.
priorminpurchase_value: Minimum value among the customer's previous purchases.
priormaxpurchase_value: Maximum value among the customer's previous purchases.
priormeangap_days: Average number of days between the customer's previous purchases.
priorstdgap_days: Standard deviation of the intervals between previous purchases, measuring variability in purchasing frequency.
4. Customer Activity and Spending Over Time
purchasecount30d: Number of purchases made during the previous 30 days.
spend_30d: Total amount spent during the previous 30 days.
purchasecount90d: Number of purchases made during the previous 90 days.
spend_90d: Total amount spent during the previous 90 days.
purchasecount180d: Number of purchases made during the previous 180 days.
spend_180d: Total amount spent during the previous 180 days.
purchasecount365d: Number of purchases made during the previous 365 days.
spend_365d: Total amount spent during the previous 365 days.


Customer and Purchase identification Variables:
29- customer_id: Unique client identifier associated with this specific transaction.
30- payment_type: Payment method used by this specific client on this specific transaction.
31- salesperson: Employee who served a client's specific transaction.
32- commercial_zone: Commercial region where this specific transaction was made.
33- transport_method: Type of transportation used for this specific purchase.
34- warehouse: Warehouse where the item for this specific purchase was stored.
35- document_series: Commercial document series used to classify this specific transaction.
36- purchase_date: Date on which this specific transaction occurred.
37- ID: Identifier of this specific transaction (or DataFrame row).


Additional Variables:
38-random_noise: A variable that appears to contain randomly generated values. Its exact purpose cannot be confirmed from the variable name alone.

The variable repeat_purchase_90d is a binary outcome variable (0 if no purchase was made within 90 days after a specific transaction, or 1 if it was done). Thus, it will become our target. This is not a feature, it is a label.

In [4]:
train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
Total a Pagar,6469.0,9.255563e+02,1.334998e+03,9.300000e-01,1.889200e+02,4.767400e+02,1.087670e+03,2.229721e+04
Total Bruto,6469.0,8.861527e+02,1.327641e+03,8.300000e-01,1.757400e+02,4.457400e+02,1.036700e+03,2.157828e+04
Total Liquido,6469.0,7.733364e+02,1.136356e+03,7.600000e-01,1.543800e+02,3.899100e+02,9.033700e+02,1.812781e+04
Total Imp_,6469.0,1.522199e+02,2.252624e+02,0.000000e+00,2.993000e+01,7.974000e+01,1.848400e+02,4.169400e+03
Total Portes,6469.0,1.151427e+00,5.844091e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.810600e+02
Total Desc_ Global,6469.0,3.499072e-01,5.116564e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.184900e+02
Total Desc_ Linha,6469.0,1.124664e+02,2.899863e+02,0.000000e+00,0.000000e+00,1.313000e+01,9.228000e+01,4.103440e+03
Custo,6469.0,3.281650e+02,5.195072e+02,0.000000e+00,4.905000e+01,1.525400e+02,3.960615e+02,7.427401e+03
N_ Detalhes,6469.0,2.152450e+01,2.771342e+01,1.000000e+00,4.000000e+00,1.100000e+01,2.900000e+01,4.230000e+02
invoice_count_current_purchase,6469.0,1.066007e+00,3.108069e-01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,5.000000e+00


In the case of qualitative categorical variables we count, per feature, the quantity of null-values, unique values, the most frequent value and its frequency.

In [5]:
train_df.describe(include="str").T

,count,unique,top,freq
payment_type,6274,9,Maybe Tomorrow Payment 7,4565
salesperson,6275,9,VLOOKUP Master 1,4054
commercial_zone,6469,5,Decorative Dashboard Zone 4,6260
transport_method,6273,80,Maybe Tomorrow Delivery 72,1119
warehouse,6274,2,Final Excel Warehouse 1,6253
document_series,6469,3,Ctrl Z Series 2,2738
purchase_date,6469,1492,2015-10-31,52


The code below creates a table indicating how many and in what proportion NA values are in the columns of our train.csv DataFrame. It is basically used to search the columns that have no missing values.

In [6]:
rows_with_null = train_df.isna().any(axis=1)
per_column = train_df.isna().sum()

pd.Series({
    "Rows with NA value": f"{rows_with_null.sum():,} of {len(train_df):,}"
                        f" ({100*rows_with_null.mean():.2f}%)",
    "Columns with NA value": f"{(per_column>0).sum()} of {train_df.shape[1]}",
    "Columns without NA values": f"{(per_column==0).sum()} of {train_df.shape[1]}",
    "NA cells": f"{train_df.isna().sum().sum():,}",
    "Column with the most NA values": f"{per_column.idxmax()} ({per_column.max()} nulls)",
}).to_frame("missing values as the file arrives")

,missing values as the file arrives
Rows with NA value,"2,605 of 6,534 (39.87%)"
Columns with NA value,36 of 39
Columns without NA values,3 of 39
NA cells,"9,818"
Column with the most NA values,prior_std_gap_days (1478 nulls)


In [7]:
print("Columns with no NA values:")
for col in train_df.columns[~train_df.isna().any()]:
    print(col)

Columns with no NA values:
repeat_purchase_90d
customer_id
ID


After knowing which 3 columns have no missing values, we need to set the index. Our goal here is to check the variable (feature) column called ID, since one purchase should have its unique ID. Verifying this following question is required: 
Is this ID unique, this is, does it not have repeted values?
If the answers is yes, then we can do it.

In [8]:
duplicated_purchase_ID=train_df.index.duplicated(keep='first').sum()
print("Number of duplicated values on the purchase ID column in train.csv:",duplicated_purchase_ID)

Number of duplicated values on the purchase ID column in train.csv: 0


In [9]:
train_df.set_index('ID', inplace=True)
print("Index name:", train_df.index.name)

Index name: ID


Guaranteed the fact that no repeat_purchase_90d value is null (NA), we proceed to the analysis of the target column. The first step is to count absolute and relative the frequency of 1's and 0's. Our aim is to discover other features patterns inside the distribution of these binary values.

In [125]:
count_target=train_df['repeat_purchase_90d'].value_counts()
target_prop=count_target.to_frame("count").assign(proportion=lambda d: d["count"]/d["count"].sum())
target_prop

,count,proportion
repeat_purchase_90d,,
1,3654,0.559229
0,2880,0.440771


# 4 · Modify the data

In this section we should decide which features seem more relevant; we examine all quantitative features grouped by their target variable, i.e the repeat_purchase_90d value. Here we can check the differences between feature values according to their repeat_purchase_90d value (0 or 1).
One ideia would be discarding all features that display significantly similiar values between repeat_purchase_90d=0 and repeat_purchase_90d=1; a candidate would be the feature variable Total Portes.

In [10]:
train_df.groupby('repeat_purchase_90d').mean(numeric_only=True).round(3).T

repeat_purchase_90d,0,1
Total a Pagar,808.556,1017.580
Total Bruto,762.809,983.165
Total Liquido,680.063,846.698
Total Imp_,128.493,170.882
Total Portes,1.156,1.148
Total Desc_ Global,0.418,0.297
Total Desc_ Linha,82.328,136.171
Custo,299.831,350.451
N_ Detalhes,20.794,22.099
invoice_count_current_purchase,1.049,1.080


Checking if Total Portes column was removed.

In [11]:
train_df.drop(['Total Portes'],axis=1,inplace=True)
train_df.head()

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,...,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,random_noise
ID,,,,,,,,,,,,,,,,,,,,,
10000000,558.11,510.90,453.75,104.36,0.0,57.15,153.099911,4.0,1.0,1,...,858.30,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,-0.660448
10000001,978.80,830.02,795.77,183.03,0.0,34.25,472.943134,32.0,1.0,1,...,2500.68,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,0.924510
10000002,10.33,12.00,8.40,1.93,0.0,3.60,6.097485,1.0,1.0,1,...,7899.80,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,-0.258730
10000003,856.83,696.61,696.61,160.22,0.0,0.00,408.167275,9.0,1.0,0,...,270.33,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,1.448402


Now we will be looking for inconsistencies and errors in this train.csv DataFrame. After identifying rows full of NA values (where customer_id and repeat_purchase_90d feature columns are ignored), we delete them from this DataFrame. We are facing variables MCAR; missing completely at random. In this case, we cannot infer any idea from then and, therefore, they are of no practical use. 
Keeping this type of missing values will only cost memory an may not work well for our future predictions.

In [4]:
drop_not_null_df=train_df.drop(columns=["customer_id", "repeat_purchase_90d"])
all_missing=drop_not_null_df.isna().all(axis=1)
train_df[all_missing]

#dropping rows filled exclusively with null values
train_df.dropna(how='all')

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise
0,558.11,510.90,453.75,104.36,0.0,0.00,57.15,153.099911,4.0,1.0,...,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,10000000,-0.660448
1,978.80,830.02,795.77,183.03,0.0,0.00,34.25,472.943134,32.0,1.0,...,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,10000001,0.924510
2,10.33,12.00,8.40,1.93,0.0,0.00,3.60,6.097485,1.0,1.0,...,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,10000002,-0.258730
3,856.83,696.61,696.61,160.22,0.0,0.00,0.00,408.167275,9.0,1.0,...,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,10000003,1.448402
4,1616.10,1488.85,1313.90,302.20,0.0,0.00,174.95,485.401864,33.0,1.0,...,122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,10000004,-1.058890
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6529,3023.34,3761.38,2458.00,565.34,0.0,0.00,1303.38,1099.190000,49.0,1.0,...,151492,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 72,Final Excel Warehouse 1,Ctrl Z Series 3,2016-01-24,10007670,-1.579719
6530,1362.26,1723.10,1107.53,254.73,0.0,34.25,581.32,197.514000,16.0,1.0,...,975873,Maybe Tomorrow Payment 12,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 1,2021-04-13,10007671,-0.576883
6531,4396.55,3574.43,3574.43,822.12,0.0,0.00,0.00,2075.838024,78.0,1.0,...,755628,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 72,Final Excel Warehouse 1,Ctrl Z Series 2,2022-03-19,10007672,0.528377
6532,452.07,483.60,367.54,84.53,0.0,0.00,116.06,265.159468,7.0,1.0,...,703834,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 40,Final Excel Warehouse 1,Ctrl Z Series 1,2019-03-23,10007673,1.012809


In [ ]:
# Ploting the distributions of the numerical features
# we want to explore the skewness of each variable
# and to understand if there are negative values where only positive numbers make sense, or extreme values far from the mean
# we also want to see if the scale of some variables is different from others to see if we will need to do scaling

num_cols = train_df.select_dtypes(include=[np.number]).columns
num_cols = [c for c in num_cols if c not in ['ID', 'repeat_purchase_90d']]

train_df[num_cols].hist(figsize=(12, 10), bins=30)
plt.suptitle("Distribution of Numerical Variables")
plt.tight_layout()
plt.show()

Uploading the clientes.csv file from the folder directly to this Jupyter notebook. As usual, we inspect its Data Frame dimension.

In [25]:
clients_df=pd.read_csv('clientes.csv')

print("shape of clientes.csv DataFrame:"clients_df.shape)

(1143, 13)

We start by analysing a specific Feature column: customer_id. Again, what we pretend to extract from it are exactly two informations: 
1# How many missing values does this column have?
2# Are there any numbers (i.e, Customer ID) duplicated?
If there is no missing values and no duplicated values in customer_id column, we can use it as an index for listing all the rows, since each of them will have an unique index.

In [43]:
null_customers=clients_df['customer_id'].isna().sum()
duplicated_customers=clients_df['customer_id'].duplicated(keep='first').sum()
print("Number of non-existent Customer ID in clients.csv:",null_customers)
print("Number of duplicated Customer ID in clients.csv:",duplicated_customers)

Number of non-existent Customer ID in clients.csv: 0
Number of duplicated Customer ID in clients.csv: 0


Indexing the customer_id column.

In [55]:
clients_df.set_index('customer_id',drop=True,append=False,inplace=False)

,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address,customer_type,company_name,customer_since_year
customer_id,,,,,,,,,,,,
842570,Country CT11: Land of The Final Version,Region RG37: We Will See Later,NaN,NaN,City CY417: Missing Cell Port,NaN,Postal code PC828: #REF! Delivery Route,NaN,Address AD1120: Cell Without a Home,NaN,Excel Is Not A Database Ltd 150,2018
770941,Country CT9: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,Postal code PC56: #REF! Delivery Route,NaN,Address AD62: Cell Without a Home,NaN,Excel Is Not A Database Ltd 1064,2021
827053,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC827: #REF! Delivery Route,Street ST170: PowerPoint New Walk,Address AD449: Cell Without a Home,NaN,Excel Is Not A Database Ltd 113,2016
494648,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD454: Cell Without a Home,NaN,Excel Is Not A Database Ltd 430,2015
429291,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD453: Cell Without a Home,NaN,Excel Is Not A Database Ltd 673,2015
...,...,...,...,...,...,...,...,...,...,...,...,...
750900,Country CT10: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Address AD1118: Cell Without a Home,NaN,Excel Is Not A Database Ltd 888,2015
516401,Country CT10: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Address AD1117: Cell Without a Home,NaN,Excel Is Not A Database Ltd 833,2015
404238,Country CT10: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Address AD1119: Cell Without a Home,NaN,Excel Is Not A Database Ltd 636,2015
